# Lumber transportation problem (J. Reeb and S. Leavengood)

Millco has three wood mills and is planning three new logging sites. Each mill has a required daily demand and each logging site can harvest a certain number of truckloads of lumber per day. Each delivery includes an equal-distance return trip; both legs cost $\$2$/mile, so a delivery costs $\$4$ times the one-way distance. If distances from logging sites to mills are given below, how should the hauls be routed to minimize hauling costs while meeting all demands?

| Logging Site | Mill A | Mill B | Mill C | Max loads per day |
|:------------:|:------:|:------:|:------:|:-----------------:|
| 1            |  8     |  15    |  50    |  20               |
| 2            |  10    |  17    |  20    |  30               |
| 3            |  30    |  26    |  15    |  45               |
| Mill demand  |  30    |  35    |  30    |                   |

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. All required packages are already included, and no external data files are needed.


### Solution

In [ ]:
# this solution uses NamedArrays, which are arrays indexed over sets for both x and y dimensions.

using JuMP, HiGHS, NamedArrays
import MathOptInterface as MOI

sites = [ 1,  2,  3]
mills = [:A, :B, :C]

cost_per_haul = 4    # don't forget the return trip!

dist = NamedArray( [8 15 50; 10 17 20; 30 26 15], (sites,mills), ("Sites","Mills") )
supply = Dict(zip( sites, [20 30 45] ))
demand = Dict(zip( mills, [30 35 30] ))

m = Model(HiGHS.Optimizer)
set_silent(m)

@variable(m, x[sites,mills] >= 0)          # x[i,j] is lumber shipped from site i to mill j.

@constraint(m, sup[i in sites], sum(x[i,j] for j in mills) == supply[i] )   # supply constraint
@constraint(m, dem[j in mills], sum(x[i,j] for i in sites) == demand[j] )   # demand constraint

@objective(m, Min, cost_per_haul*sum( x[i,j]*dist[i,j] for i in sites, j in mills ) )      # minimize transportation cost

optimize!(m)
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end

status=termination_status(m)
println(status)

# nicely formatted solution
solution = NamedArray( [round(value(x[i,j]); digits=6) for i in sites, j in mills], (sites,mills), ("Sites","Mills") )
println( solution )
println()
println("Total cost will be \$", objective_value(m))

### Compact version of the problem

In [ ]:
using JuMP, HiGHS, LinearAlgebra
import MathOptInterface as MOI

m = Model(HiGHS.Optimizer)
set_silent(m)

# incidence matrix:
A = [ 1  1  1  0  0  0  0  0  0
      0  0  0  1  1  1  0  0  0
      0  0  0  0  0  0  1  1  1
     -1  0  0 -1  0  0 -1  0  0
      0 -1  0  0 -1  0  0 -1  0
      0  0 -1  0  0 -1  0  0 -1 ]

# supply and demand
b = [ 20, 30, 45, -30, -35, -30 ]

# distances
c = [ 8, 15, 50, 10, 17, 20, 30, 26, 15, ]

@variable(m, x[1:9] >= 0)
@constraint(m, A*x .== b)
@objective(m, Min, 4*dot(c,x))

optimize!(m)
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end
xsol = value.(x)
display( reshape(xsol,3,3)' )

### Minimum-cost network-flow version

Use the same algebraic model as in [12-mcnf.ipynb](12-mcnf.ipynb), with node set `N` and arc set `E`. Logging sites 1, 2, and 3 are sources; mills A, B, and C are sinks. Every site can ship to every mill.

Positive $b_i$ denotes supply and negative $b_i$ denotes demand. Total supply equals total demand, so all site supply is used. Costs $c_{ij}$ include the return trip; arc bounds are $\ell_{ij}=0$ and $u_{ij}=\infty$.


In [ ]:
N = [1, 2, 3, :A, :B, :C]
E = [(i,j) for i in [1, 2, 3] for j in [:A, :B, :C]]

# Dollars per truckload: $2/mile on both legs of the round trip.
c = Dict(zip(E, 4 .* [8, 15, 50, 10, 17, 20, 30, 26, 15]))

# Net supplies (truckloads/day), in the same order as N.
b = Dict(zip(N, [20, 30, 45, -30, -35, -30]))

# Arc bounds (truckloads/day).
ℓ = Dict((i,j) => 0 for (i,j) in E)
u = Dict((i,j) => Inf for (i,j) in E)


#### Model


In [ ]:
using JuMP, HiGHS
import MathOptInterface as MOI

m = Model()

@variable(m, ℓ[i,j] <= x[(i,j) in E] <= u[i,j])

@objective(m, Min,
    sum(c[i,j]x[(i,j)] for (i,j) in E))

@constraint(m, flowbalance[i in N],
    sum(x[(i,j)] for j in N if (i,j) in E) -
    sum(x[(j,i)] for j in N if (j,i) in E)
    == b[i])

set_optimizer(m, HiGHS.Optimizer)
set_silent(m)
optimize!(m)


#### Solution


In [ ]:
println("Status: ", termination_status(m))
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end

println("Minimum cost = ", round(objective_value(m), digits=1), " dollars/day")
for (i,j) in E
    if value(x[(i,j)]) > 0.00001
        println("Flow from node ", i, " to node ", j, " = ",
            round(value(x[(i,j)]), digits=1), " truckloads/day")
    end
end
